In [ ]:
%pip install -q MetaTrader5 statsmodels arch matplotlib

In [ ]:
# =====================================================================
#  CONFIG  - the ONLY cell you need to edit.   (Open MT5 and log in to your demo account first.)
# =====================================================================
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from arch import arch_model
import MetaTrader5 as mt5
from IPython.display import display, clear_output

# ---- MT5 ----
MT5_SYMBOL = "XAUUSDm"          # EXACTLY as shown in MT5 > Market Watch (e.g. "BTCUSDm")
MT5_LOGIN, MT5_PASSWORD, MT5_SERVER = None, None, None    # None = use the account already logged in inside MT5

# ---- model ----
HORIZON_MIN = 15                # forecast the move over the next N minutes (5 to 30 makes sense for scalping)
TRAIN_DAYS  = 20                # trading days of 1-minute bars used to fit the model
TEST_DAYS   = 10                # trading days used by the Step 4 check (walk-forward test)

# ---- live loop ----
POLL_SECONDS    = 5             # screen refresh
REFIT_EVERY_MIN = 60            # refit the model every N minutes
LOG_ROWS        = 10            # rows in the live table

# ---- fixed design (no need to change) ----
W_SHORT, W_MID, W_LONG = 15, 60, 1380    # HAR parts: last 15 minutes, last hour, last trading day (~23 h)
BUCKET = 15                              # time-of-day pattern measured in 15-minute slots
EPS = 1e-3

In [ ]:
# ---------------------------------------------------------------------
#  Helper functions (run once, nothing to edit)
# ---------------------------------------------------------------------
def mt5_connect():
    ok = mt5.initialize() if MT5_LOGIN is None else mt5.initialize(login=MT5_LOGIN, password=MT5_PASSWORD, server=MT5_SERVER)
    if not ok:
        raise RuntimeError(f"MT5 initialize failed: {mt5.last_error()}  -> is MT5 open and logged in to your demo account?")
    if not mt5.symbol_select(MT5_SYMBOL, True):
        raise RuntimeError(f"Symbol '{MT5_SYMBOL}' not found: {mt5.last_error()}  -> copy the exact name from Market Watch")
    print("Connected to MT5:", getattr(mt5.account_info(), "server", "?"), "| symbol:", MT5_SYMBOL)


def mt5_m1(n):
    """Last n CLOSED 1-minute bars (position 1 skips the still-forming bar). Index = broker server time."""
    rates = mt5.copy_rates_from_pos(MT5_SYMBOL, mt5.TIMEFRAME_M1, 1, n)
    if rates is None or len(rates) == 0:
        raise RuntimeError(f"No bars returned: {mt5.last_error()}")
    d = pd.DataFrame(rates)
    d["time"] = pd.to_datetime(d["time"], unit="s")
    d = d.set_index("time").sort_index()
    return d[~d.index.duplicated(keep="last")]["close"].dropna()


def clean_returns(close):
    """1-minute log returns; the return across any gap (weekend, daily break) is dropped."""
    r = np.log(close).diff()
    gap = close.index.to_series().diff() > pd.Timedelta(minutes=5)
    return r[~gap].dropna()


def slot(index):
    """15-minute slot of the day (0 ... 95) for each timestamp."""
    return ((index.hour * 60 + index.minute) // BUCKET).to_numpy()


def tod_factor(r):
    """Time-of-day pattern: how busy each 15-minute slot usually is, relative to the average minute (1.0 = average).
    Built from mean absolute returns (robust to spikes), lightly smoothed with the neighbouring slots."""
    m = r.abs().groupby(slot(r.index)).mean().reindex(range(1440 // BUCKET)).interpolate(limit_direction="both")
    m = (m.shift(1, fill_value=m.iloc[-1]) + 2 * m + m.shift(-1, fill_value=m.iloc[0])) / 4
    return (m / r.abs().mean()) ** 2


def future_slots_sum(index, s, h):
    """Sum of the time-of-day factor over the next h minutes after each timestamp (clock based)."""
    mins = (index.hour * 60 + index.minute).to_numpy()
    sv, out = s.to_numpy(), np.zeros(len(index))
    for k in range(1, h + 1):
        out += sv[((mins + k) % 1440) // BUCKET]
    return out


def har_parts(r, s, scale):
    """Squared returns with the time-of-day pattern taken out, and the three HAR parts (15 min, 1 hour, 1 day)."""
    a = pd.Series(r.to_numpy() ** 2 / (s.to_numpy()[slot(r.index)] * scale), index=r.index)
    X = pd.DataFrame({"short": a.rolling(W_SHORT).mean(), "mid": a.rolling(W_MID).mean(), "long": a.rolling(W_LONG).mean()})
    return a, X


def ahead(x, h):
    """Sum of x over the next h rows (the target / the outcome)."""
    return x[::-1].rolling(h).sum()[::-1].shift(-1)


class IntradayHAR:
    """log-HAR-RV on 1-minute data with a time-of-day adjustment. Forecasts the variance of the next h minutes."""

    def __init__(self, h, use_tod=True):
        self.h, self.use_tod = h, use_tod

    def fit(self, r):
        self.s = tod_factor(r) if self.use_tod else pd.Series(1.0, index=range(1440 // BUCKET))
        self.scale = float((r ** 2).mean())
        a, X = har_parts(r, self.s, self.scale)
        y = ahead(a, self.h) / self.h
        d = pd.concat([X, y.rename("y")], axis=1).dropna().iloc[::self.h]     # non-overlapping targets
        self.res = sm.OLS(np.log(d["y"] + EPS), sm.add_constant(np.log(d[["short", "mid", "long"]] + EPS))).fit(
            cov_type="HAC", cov_kwds={"maxlags": 4})
        self.smear = float(np.mean(np.exp(self.res.resid)))                   # back-transform correction for the log
        V = self.variance(X.loc[d.index])
        z = (ahead(r, self.h).loc[d.index] / np.sqrt(V)).dropna()
        self.q68, self.q95 = float(z.abs().quantile(0.68)), float(z.abs().quantile(0.95))   # range widths from real moves
        self.n_obs, self.fitted_at = len(d), r.index[-1]
        return self

    def variance(self, X):
        """Forecast variance (log-return units) of the move over the next h minutes, for each row of X."""
        Xl = sm.add_constant(np.log(X[["short", "mid", "long"]] + EPS), has_constant="add")
        level = np.exp(Xl.to_numpy() @ self.res.params.to_numpy()) * self.smear
        return pd.Series(level * future_slots_sum(X.index, self.s, self.h) * self.scale, index=X.index)


def garch_variance(r_train, r_window, h):
    """Baseline for Step 4: GARCH(1,1)-t on 1-minute returns, variance of the next h minutes."""
    k = 1 / r_train.std()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        g = arch_model(r_train * k, mean="Zero", vol="GARCH", p=1, q=1, dist="t", rescale=False).fit(disp="off")
        gf = arch_model(r_window * k, mean="Zero", vol="GARCH", p=1, q=1, dist="t", rescale=False).fix(g.params)
    om, al, be = g.params["omega"], g.params["alpha[1]"], g.params["beta[1]"]
    p = min(al + be, 0.99999)
    wbar = om / (1 - p)
    h1 = om + al * (r_window * k).to_numpy() ** 2 + be * gf.conditional_volatility.to_numpy() ** 2
    return pd.Series((h * wbar + (h1 - wbar) * (1 - p ** h) / (1 - p)) / k ** 2, index=r_window.index)


def full_days(r, min_bars=1000):
    cnt = r.groupby(r.index.floor("D")).count()
    return cnt[cnt >= min_bars].index


def to_ist(ts):
    return ts + pd.Timedelta(hours=5, minutes=30)      # broker time is UTC for Exness; adjust if your broker differs

In [ ]:
# ==============================================================================
# STEP 1: LOAD 1-MINUTE HISTORY FROM MT5 (closed bars only)
# ==============================================================================
mt5_connect()
close = mt5_m1((TRAIN_DAYS + TEST_DAYS + 3) * 1440)
mt5.shutdown()

r = clean_returns(close)
days = full_days(r)
print(f"{MT5_SYMBOL} M1: {len(close):,} closed bars | {close.index[0]}  ->  {close.index[-1]}  (broker time)")
print(f"Full trading days: {len(days)}  (need at least {TRAIN_DAYS + 2} to fit, {TRAIN_DAYS + TEST_DAYS} for the Step 4 test)")
if len(days) < TRAIN_DAYS + 2:
    print("-> Not enough history: lower TRAIN_DAYS, or raise 'Max bars in chart' in MT5 (Tools > Options > Charts).")

In [ ]:
# ==============================================================================
# STEP 2: THE TIME-OF-DAY PATTERN (how busy each 15 minutes of the day usually is)
# ==============================================================================
r_train = r[r.index >= days[-TRAIN_DAYS]]
s = tod_factor(r_train)
busiest, quietest = int(s.idxmax()), int(s.idxmin())
hm = lambda k: f"{k * BUCKET // 60:02d}:{k * BUCKET % 60:02d}"
ist = lambda k: (pd.Timestamp("2000-01-01") + pd.Timedelta(minutes=k * BUCKET + 330)).strftime("%H:%M")

plt.figure(figsize=(12, 4.5))
plt.bar(s.index, s.values, width=0.85, color=["#2563eb" if k == busiest else "#b8b8b0" for k in s.index])
plt.axhline(1, color="grey", linestyle="--", linewidth=1)
plt.text(len(s) - 1, 1.05, "average minute = 1.0", ha="right", color="grey")
ticks = list(range(0, len(s), 8))
plt.xticks(ticks, [f"{hm(k)}\n{ist(k)} IST" for k in ticks], fontsize=8)
plt.title(f"{MT5_SYMBOL}: busiest at {hm(busiest)} broker time ({ist(busiest)} IST), "
          f"{s.max() / s.min():.1f}x the quietest slot ({hm(quietest)})")
plt.ylabel("Variance vs the average minute"); plt.grid(axis="y", linestyle=":", alpha=0.6); plt.tight_layout(); plt.show()

print(f"Busiest 15 minutes : {hm(busiest)} broker = {ist(busiest)} IST  ->  {s.max():.2f}x an average minute")
print(f"Quietest 15 minutes: {hm(quietest)} broker = {ist(quietest)} IST  ->  {s.min():.2f}x an average minute")
print("The model takes this pattern out before fitting and puts it back into every forecast.")

In [ ]:
# ==============================================================================
# STEP 3: FIT THE INTRADAY HAR-RV MODEL (log-HAR + time of day)
# ==============================================================================
model = IntradayHAR(HORIZON_MIN).fit(r_train)
p = model.res.params
table = pd.DataFrame({"Coefficient": p.round(3), "t-stat": model.res.tvalues.round(2)})
table.index = ["Constant", "Last 15 min", "Last hour", "Last day"]
print(f"Fitted on {TRAIN_DAYS} trading days ({model.n_obs:,} non-overlapping {HORIZON_MIN}-minute periods), log scale:")
display(table)
print(f"Persistence (sum of the three weights): {p.iloc[1:].sum():.2f}   (closer to 1 = volatility shocks last longer)")
print(f"Range widths from real moves: 68% range = ±{model.q68:.2f} sigma, 95% range = ±{model.q95:.2f} sigma"
      f"   (a normal curve would say 1.00 and 1.96)")

a_now, X_now = har_parts(r, model.s, model.scale)
V_now = float(model.variance(X_now.dropna().iloc[[-1]]).iloc[0])
last = float(close.iloc[-1])
print(f"\nForecast from the last closed bar ({close.index[-1]}): the next {HORIZON_MIN} minutes move about "
      f"±{last * np.sqrt(V_now):.3f} (1 sigma), 95% range {last * np.exp(-model.q95 * np.sqrt(V_now)):.3f} ... "
      f"{last * np.exp(model.q95 * np.sqrt(V_now)):.3f}")

In [ ]:
# ==============================================================================
# STEP 4: WALK-FORWARD TEST ON YOUR DATA (fit on the past only, score on the next day, repeat)
#   compares: HAR + time of day | HAR without time of day | GARCH(1,1) on 1-minute bars | last-15-minutes (naive)
# ==============================================================================
H = HORIZON_MIN
test_days = days[-TEST_DAYS:] if len(days) >= TRAIN_DAYS + TEST_DAYS else days[TRAIN_DAYS:]
rows = []
for day in test_days:
    past = days[days < day][-TRAIN_DAYS:]
    rtr = r[(r.index >= past[0]) & (r.index < day)]
    win = r[(r.index >= past[0]) & (r.index < day + pd.Timedelta(days=1))]
    is_test, rv_next, move_next = win.index >= day, ahead(win ** 2, H), ahead(win, H)
    cands = {}
    for name, tod in [("HAR + time of day", True), ("HAR, no time of day", False)]:
        m = IntradayHAR(H, tod).fit(rtr)
        cands[name] = m.variance(har_parts(win, m.s, m.scale)[1])
    cands["GARCH(1,1) on 1-min bars"] = garch_variance(rtr, win, H)
    cands[f"Naive (last {H} min)"] = (win ** 2).rolling(H).sum()
    for name, V in cands.items():
        z = (move_next / np.sqrt(V))[~is_test].replace([np.inf, -np.inf], np.nan).dropna()
        z = z[z.index.minute % H == 0].abs()                      # range widths from this model's own past moves
        rows.append(pd.DataFrame({"model": name, "V": V, "rv": rv_next, "move": move_next,
                                  "q68": z.quantile(0.68), "q95": z.quantile(0.95)})[is_test])
R = pd.concat(rows).dropna()
R = R[(R.V > 0) & (R.rv > 0) & (R.index.minute % H == 0)]        # non-overlapping test periods
R["Session (broker time)"] = pd.cut(R.index.hour, [-1, 6, 12, 19, 23], labels=["Asia 00-07", "Europe 07-13", "US 13-20", "Late 20-24"])

def score(g):
    x = g.rv / g.V
    return pd.Series({"QLIKE (lower = better)": (x - np.log(x) - 1).mean(),
                      "Correlation with real": np.corrcoef(np.log(g.V), np.log(g.rv))[0, 1],
                      "Inside 68% range": (g.move.abs() <= g.q68 * np.sqrt(g.V)).mean(),
                      "Inside 95% range": (g.move.abs() <= g.q95 * np.sqrt(g.V)).mean()})

summary = R.groupby("model").apply(score).sort_values("QLIKE (lower = better)")
print(f"{len(test_days)} test days ({test_days[0].date()} to {test_days[-1].date()}), {R.groupby('model').size().iloc[0]:,} "
      f"non-overlapping {H}-minute periods per model:")
shown = summary.copy()
shown["QLIKE (lower = better)"] = shown["QLIKE (lower = better)"].map("{:.3f}".format)
shown["Correlation with real"] = shown["Correlation with real"].map("{:.2f}".format)
for col in ["Inside 68% range", "Inside 95% range"]:
    shown[col] = shown[col].map("{:.0%}".format)
display(shown)
by_sess = R.groupby(["model", "Session (broker time)"], observed=True).apply(
    lambda g: (g.move.abs() <= g.q95 * np.sqrt(g.V)).mean()).unstack()
print("\n95% range: how often the real move stayed inside, by session (target 95% everywhere):")
display(by_sess.apply(lambda col: col.map("{:.0%}".format)))
best = summary.index[0]
print(f"\nBest forecaster on your data: {best}.")

In [ ]:
# ---------------------------------------------------------------------
#  MT5: LIVE loop - intraday HAR-RV for scalping (closed 1-minute bars only)
#   - every new closed minute: forecast how far price can move in the next HORIZON_MIN minutes
#   - the model is refitted every REFIT_EVERY_MIN minutes on the last TRAIN_DAYS days
#   - the screen refreshes every POLL_SECONDS with the live price
#   - a tracker checks every forecast HORIZON_MIN minutes later: did the real move stay inside the range?
#  Stop with the notebook's stop button.
# ---------------------------------------------------------------------
LIVE_LOG = {}                    # one row per closed minute, newest last
PENDING = []                     # forecasts waiting for their outcome
TRACK = {"n": 0, "in68": 0, "in95": 0}


def fit_live():
    c = mt5_m1((TRAIN_DAYS + 3) * 1440)
    rr = clean_returns(c)
    dd = full_days(rr)
    return IntradayHAR(HORIZON_MIN).fit(rr[rr.index >= dd[-min(TRAIN_DAYS, len(dd))]])


def forecast_now(m, tail):
    """Forecast from the last closed bar: variance of the next h minutes + how it builds minute by minute."""
    rr = clean_returns(tail)
    a, X = har_parts(rr, m.s, m.scale)
    x_last = X.iloc[[-1]]
    V = float(m.variance(x_last).iloc[0])
    t = tail.index[-1]
    mins = t.hour * 60 + t.minute
    s_next = np.array([m.s.iloc[((mins + k) % 1440) // BUCKET] for k in range(1, m.h + 1)])
    path = V * np.cumsum(s_next) / s_next.sum()                     # variance builds up with the time-of-day pattern
    busy = float(s_next.mean())                                       # next h minutes vs an average minute
    activity = float(x_last["short"].iloc[0])                         # last 15 minutes vs usual for that time of day
    return V, path, busy, activity


def check_outcomes(tail):
    """Score forecasts whose h minutes have now passed."""
    pos = {t: i for i, t in enumerate(tail.index)}
    keep = []
    for f in PENDING:
        i = pos.get(f["time"])
        if i is None or i + f["h"] >= len(tail):
            keep.append(f); continue
        t_end = tail.index[i + f["h"]]
        if t_end - f["time"] > pd.Timedelta(minutes=f["h"] + 5):      # a gap in between: not a fair test, skip
            continue
        move = abs(np.log(tail.iloc[i + f["h"]] / f["close"]))
        in68, in95 = move <= f["q68"] * np.sqrt(f["V"]), move <= f["q95"] * np.sqrt(f["V"])
        TRACK["n"] += 1; TRACK["in68"] += int(in68); TRACK["in95"] += int(in95)
        if f["time"] in LIVE_LOG:
            LIVE_LOG[f["time"]]["Result"] = "inside 68%" if in68 else "inside 95%" if in95 else "OUTSIDE 95%"
    PENDING[:] = keep


def get_tick():
    tick = mt5.symbol_info_tick(MT5_SYMBOL)
    if tick is None or tick.bid <= 0:
        return None
    return {"bid": tick.bid, "ask": tick.ask, "live": (tick.bid + tick.ask) / 2, "time": pd.to_datetime(tick.time, unit="s")}


def render(m, tail, V, path, busy, activity, d, stale):
    last, t = float(tail.iloc[-1]), tail.index[-1]
    sd = np.sqrt(V)
    lo68, hi68 = last * np.exp(-m.q68 * sd), last * np.exp(m.q68 * sd)
    lo95, hi95 = last * np.exp(-m.q95 * sd), last * np.exp(m.q95 * sd)
    tk = get_tick()

    # ---- chart first ----
    hist = tail.iloc[-120:]
    xh = np.arange(-len(hist) + 1, 1)
    xf = np.arange(0, m.h + 1)
    sp = np.sqrt(np.r_[0, path])
    plt.figure(figsize=(12, 5))
    plt.plot(xh, hist.values, color="navy", label="Price (closed 1-minute bars)")
    plt.fill_between(xf, last * np.exp(-m.q95 * sp), last * np.exp(m.q95 * sp), color="#93c5fd", alpha=0.35, label="95% range")
    plt.fill_between(xf, last * np.exp(-m.q68 * sp), last * np.exp(m.q68 * sp), color="#2563eb", alpha=0.35, label="68% range")
    if tk is not None and not stale:
        plt.scatter([0.5], [tk["live"]], color="green", s=50, zorder=5, label="Live price")
    plt.axvline(0, color="grey", linewidth=0.8, linestyle=":")
    plt.title(f"{MT5_SYMBOL}: next {m.h} minutes, 95% range {lo95:.{d}f} – {hi95:.{d}f}")
    plt.xlabel("Minutes (0 = last closed bar)"); plt.ylabel("Price")
    plt.grid(True, linestyle=":", alpha=0.6); plt.legend(loc="upper left"); plt.tight_layout(); plt.show(); plt.close()

    # ---- live text BELOW the chart ----
    mood = "ACTIVE" if activity > 1.5 else "CALM" if activity < 0.67 else "NORMAL"
    print(f"Refreshed at {time.strftime('%H:%M:%S')} (this computer)   |   model fitted on data up to {m.fitted_at} "
          f"(refits every {REFIT_EVERY_MIN} min)")
    print(f"{MT5_SYMBOL} M1  last closed bar: {t} broker = {to_ist(t):%H:%M} IST   close: {last:.{d}f}")
    if tk is None:
        print("Live price: not available right now")
    elif stale:
        print(f"Live price: {tk['live']:.{d}f}  (no new bar for over 3 minutes - market closed or feed paused?)")
    else:
        spread = tk["ask"] - tk["bid"]
        print(f"LIVE price : {tk['live']:.{d}f}   (bid {tk['bid']:.{d}f} / ask {tk['ask']:.{d}f}, spread {spread:.{d}f})")
        print(f"Spread check: the spread is {spread / (last * sd):.0%} of a typical {m.h}-minute move (1 sigma) "
              f"- the higher this is, the more of each move the spread eats")
    print(f"Next {m.h} min (from last close): typical move ±{last * sd:.{d}f} (1 sigma)  |  "
          f"68% range {lo68:.{d}f} – {hi68:.{d}f}  |  95% range {lo95:.{d}f} – {hi95:.{d}f}")
    print(f"Time of day : the next {m.h} minutes are usually {busy:.1f}x as busy as an average minute")
    print(f"Market now  : the last 15 minutes were {activity:.1f}x as active as usual for this time of day -> {mood}")
    if TRACK["n"]:
        n = TRACK["n"]
        print(f"Tracker ({n} forecasts checked): real move inside 68% range {TRACK['in68'] / n:.0%} (target 68%)  |  "
              f"inside 95% range {TRACK['in95'] / n:.0%} (target 95%)")
    else:
        print(f"Tracker: the first check comes {m.h} minutes after the first forecast ...")
    rows = list(LIVE_LOG.values())[-LOG_ROWS:]
    print(f"\nLast {len(rows)} closed minutes (Result = where the real move ended {m.h} minutes later):")
    display(pd.DataFrame(rows).set_index("Bar time").iloc[::-1])


def run_live():
    mt5_connect()
    info = mt5.symbol_info(MT5_SYMBOL)
    d = int(getattr(info, "digits", 3) or 3)
    print(f"Fitting the model on the last {TRAIN_DAYS} trading days ...")
    m = fit_live()
    last_seen, state, last_new = None, None, time.time()
    try:
        while True:
            tail = mt5_m1(W_LONG + 400)
            t = tail.index[-1]
            if t != last_seen:                                          # a NEW minute has closed
                last_seen, last_new = t, time.time()
                if t - m.fitted_at >= pd.Timedelta(minutes=REFIT_EVERY_MIN):
                    print("Refitting the model on fresh data ..."); m = fit_live()
                check_outcomes(tail)
                V, path, busy, activity = forecast_now(m, tail)
                last = float(tail.iloc[-1])
                PENDING.append({"time": t, "close": last, "V": V, "q68": m.q68, "q95": m.q95, "h": m.h})
                LIVE_LOG[t] = {"Bar time": t, "Close": round(last, d), f"1 sigma ({m.h} min)": round(last * np.sqrt(V), d),
                               "95% low": round(last * np.exp(-m.q95 * np.sqrt(V)), d),
                               "95% high": round(last * np.exp(m.q95 * np.sqrt(V)), d),
                               "Busy (time of day)": f"{busy:.1f}x", "Activity now": f"{activity:.1f}x", "Result": "pending"}
                while len(LIVE_LOG) > 500:
                    LIVE_LOG.pop(next(iter(LIVE_LOG)))
                state = (V, path, busy, activity)
            clear_output(wait=True)
            render(m, tail, *state, d, time.time() - last_new > 180)
            time.sleep(POLL_SECONDS)
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        mt5.shutdown()


run_live()